# 05 · When RAG Hallucinates (the honest crux)

The belief RAG does not earn. On the **absence questions**, whose answers are nowhere in the
collection, a default RAG system produces confident, fluent, **fabricated** answers. RAG
reduced hallucination by giving the model a source to lean on — it did not remove it.

We show the fabrications directly — including answers that quote the retrieved passages, so
a lexical groundedness check rates them as *supported* — and then sweep a groundedness
threshold to show how far the abstention floor can be raised, and what it costs in coverage.


In [1]:
import os, sys, json
from pathlib import Path

# Run from the repo root so outputs/ and data/ resolve no matter where the notebook starts.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import support
from src.config import load_settings
from src.datasets.corpus import build_corpus
from src.pipeline.embed import build_embedder

settings = load_settings()
corpus = build_corpus()
embedder = build_embedder(settings.embed)
generator, mode = support.select_generator(corpus, settings)
print("generator mode:", mode)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

generator mode: ollama:qwen2.5:0.5b


### Generate once on absence and answerable questions (default system)

In [2]:
from src.datasets.corpus import chunk_corpus
from src.pipeline.chunk import make_chunk_fn
from src.pipeline.retrieve import build_retriever
from src.pipeline.rag import RagPipeline

cf = make_chunk_fn(settings.chunk.strategy, size=settings.chunk.size,
                   overlap=settings.chunk.overlap, encoder=embedder)
chunks, _ = chunk_corpus(corpus, cf)
retriever = build_retriever(settings.retrieve.method, embedder, settings.retrieve.hybrid_alpha)
pipe = RagPipeline(retriever=retriever, generator=generator, settings=settings, abstain=False)
pipe.index(chunks)

absence = [(q, pipe.answer(q.text)) for q in corpus.questions_by_kind("absence")]
answerable = [(q, pipe.answer(q.text)) for q in corpus.questions_by_kind("answerable")]


### Hallucination with a footnote

Look at the absence answers. The model does not say "I don't know" — it fabricates, and
often does so **using words straight from the retrieved passages**, so a lexical
groundedness check scores the lie as *well-supported*. A high groundedness number next to a
false answer is the honest crux: retrieval gave the hallucination a source to point at.


In [3]:
from src.grounding.groundedness import groundedness_score
from src.grounding.abstention import is_abstention

footnote = [{"question": q.text, "fabricated_answer": r.raw_answer,
             "groundedness": round(groundedness_score(r.raw_answer, r.contexts), 2),
             "model_self_abstained": is_abstention(r.raw_answer)}
            for q, r in absence]
footnote_df = pd.DataFrame(footnote)
footnote_df


,question,fabricated_answer,groundedness,model_self_abstained
0,What is the population of Myrelink?,"The population of Myrelink is 48,000.",1.0,False
1,Which island in the archipelago has an interna...,The island in the archipelago with an internat...,0.6,False
2,How much does a ferry ticket between the islan...,The documents do not contain this information.,0.0,True
3,What is the average summer temperature on Sunc...,The documents do not contain this information.,0.0,True


### The abstention lever — raise the groundedness threshold

A groundedness check *does* help, but it trades coverage for safety: as the threshold
rises, the system abstains more on absence questions (good) but eventually starts refusing
answerable ones too (bad). We sweep the threshold and measure both. (Generation is
temperature 0, so the answers are fixed; only the abstain decision moves.)


In [4]:
from src.grounding.abstention import should_abstain, ABSTENTION_TEXT
from src.evaluation.answer_metrics import answer_correct

rst = settings.grounding.retrieval_score_threshold
thresholds = [0.0, 0.25, 0.5, 0.75, 1.0]
rows = []
for t in thresholds:
    abst = [should_abstain(r.raw_answer, r.contexts, r.scores,
                           retrieval_score_threshold=rst, groundedness_threshold=t)
            for _, r in absence]
    correct = 0
    for q, r in answerable:
        refuse = should_abstain(r.raw_answer, r.contexts, r.scores,
                                retrieval_score_threshold=rst, groundedness_threshold=t)
        correct += answer_correct(ABSTENTION_TEXT if refuse else r.raw_answer, q.answer)
    rows.append({"groundedness_threshold": t,
                 "abstention_on_absence": sum(abst) / len(abst),
                 "accuracy_on_answerable": correct / len(answerable)})
threshold_df = pd.DataFrame(rows)
# Abstention on absence can only rise as the threshold rises.
assert (threshold_df["abstention_on_absence"].diff().dropna() >= -1e-9).all()
threshold_df


,groundedness_threshold,abstention_on_absence,accuracy_on_answerable
0,0.00,0.50,1.0
1,0.25,0.50,1.0
2,0.50,0.50,1.0
3,0.75,0.75,1.0
4,1.00,0.75,1.0


### Figure → PNG, results → JSON

In [5]:
from src.visualisation import grouped_lines

fig = grouped_lines(thresholds,
                    {"abstention on absence (safety)": threshold_df["abstention_on_absence"],
                     "accuracy on answerable (coverage)": threshold_df["accuracy_on_answerable"]},
                    title="The abstention lever: safety vs coverage",
                    xlabel="groundedness threshold", ylabel="rate")
p = support.save_fig(fig, "05_abstention_tradeoff")
support.save_json({"mode": mode,
                   "fabrications": footnote,
                   "threshold_sweep": threshold_df.to_dict(orient="records")},
                  "05_hallucination")
print("saved", p)
fig


saved outputs/figures/05_abstention_tradeoff.png


<Figure size 700x400 with 1 Axes>